# **Intelligent Prompt Engineering Studio**

This notebook demonstrates, interactively, how six different prompting
strategies affect LLM output on the same query:

1. Zero-Shot Prompting
2. One-Shot Prompting
3. Few-Shot Prompting
4. Chain-of-Thought Prompting
5. Role-Based / Persona Prompting
6. Structured Output Prompting (JSON/Table)

## 1. Setup

In [67]:
!pip install -q transformers==4.46.3 torch gradio sentencepiece accelerate

### 2. MODEL SETUP

In [69]:
import time
import json
from transformers import pipeline
from google.colab import userdata
userdata.get('HF_AK')

MODEL_ID = "google/flan-t5-base"
text_generator = pipeline("text2text-generation", model=MODEL_ID)

def text_generation(query_input, temp=0.7, max_tokens=150, p=0.9, k=50):
    """Generate text with specified parameters.

    Args:
        query_input: The prompt/query to generate from
        temp: Temperature (0.0-1.0) - Lower=deterministic, Higher=creative
        max_tokens: Maximum length of generated response
        p: Top-p (nucleus) sampling parameter
        k: Top-k sampling parameter

    Returns:
        tuple: (generated_text, inference_time_seconds)
    """
    begin_time = time.time()
    try:
        response = text_generator(
            query_input,
            max_length=max_tokens,
            do_sample=temp > 0,
            temperature=temp,
            top_p=p,
            top_k=k,
        )[0]["generated_text"]
        elapsed = time.time() - begin_time
        return response, round(elapsed, 4)
    except Exception as ex:
        error_msg = f"[Generation failed: {str(ex)}]"
        elapsed = time.time() - begin_time
        return error_msg, round(elapsed, 4)


### 3. Prompting Strategies

In [70]:
def format_response(input_query, **options):
    """Format response request."""
    return input_query

def one_shot_example(query):
    """One-Shot: Include one example before the actual query."""
    return f"""Example:
Q: What is photosynthesis?
A: Photosynthesis is the process by which plants convert sunlight into chemical energy.

Now answer this question: {query}"""

def few_shot_examples(query):
    """Few-Shot: Include multiple examples before the actual query."""
    return f"""Examples:
Example 1: Q: What is gravity? A: Gravity is a fundamental force that attracts objects with mass toward each other.
Example 2: Q: What is electricity? A: Electricity is the flow of charged particles through a conductor.
Example 3: Q: What is motion? A: Motion is the change in position of an object over time relative to a reference point.

Now answer this question: {query}"""

def chain_of_thought(query):
    """Chain-of-Thought: Encourage step-by-step reasoning."""
    return f"""{query}

Think through this step by step:
1. First, consider the main concepts involved
2. Then, explain how they relate
3. Finally, provide your complete answer"""

def role_based(query):
    """Role-Based/Persona: Give the model a specific role."""
    return f"""You are an expert analyst with deep domain knowledge.

Question: {query}

Provide a comprehensive, expert-level answer:"""

def structured_output(query):
    """Structured Output: Ask the model to respond in a structured JSON/table format."""
    return f"""Answer the following question and format your ENTIRE response strictly as a
JSON object with exactly two keys: "answer" (a concise explanation) and "key_points"
(a list of 3 short bullet points).

Question: {query}

Respond ONLY with valid JSON. Do not include any text outside the JSON object."""

STRATEGIES = {
    "Zero-Shot": format_response,
    "One-Shot": one_shot_example,
    "Few-Shot": few_shot_examples,
    "Chain-of-Thought": chain_of_thought,
    "Role-Based / Persona": role_based,
    "Structured Output": structured_output,
}

print(f"✅ Loaded {len(STRATEGIES)} prompting strategies")

✅ Loaded 6 prompting strategies


### 4. Run All Strategies and Collect Results

In [75]:
def evaluate_response(response_text, query_text=""):
    """Evaluate response quality on multiple dimensions.

    Evaluates on: Accuracy, Length/Completeness, Clarity, Structure, Reasoning, Creativity

    Args:
        response_text: The generated response to evaluate
        query_text: The original query, used to score Accuracy (relevance)

    Returns:
        dict: Comprehensive evaluation metrics
    """

    # Accuracy evaluation (relevance heuristic): how many meaningful keywords
    # from the query actually show up in the response. This is a lightweight
    # proxy for accuracy in the absence of a ground-truth answer / LLM judge.
    stopwords = {
        "the", "is", "are", "a", "an", "of", "to", "in", "and", "what", "how",
        "why", "does", "do", "for", "on", "with", "this", "that", "your",
    }
    query_keywords = {
        w.strip(".,?!:;\"'").lower()
        for w in query_text.split()
        if len(w) > 3 and w.strip(".,?!:;\"'").lower() not in stopwords
    }
    response_lower = response_text.lower()
    if query_keywords:
        matched = sum(1 for w in query_keywords if w in response_lower)
        accuracy_score = min(10, (matched / len(query_keywords)) * 10)
    else:
        accuracy_score = 5.0

    # Length evaluation (Completeness)
    word_count = len(response_text.split())
    completeness_score = min(10, (word_count / 30) * 10)

    # Clarity evaluation (check for common quality markers)
    clarity_markers = [
        "the", "is", "are", "because", "therefore", "thus", "for example"
    ]
    marker_count = sum(1 for marker in clarity_markers if marker in response_text.lower())
    clarity_score = min(10, (marker_count / 5) * 10)

    # Structure evaluation (look for logical breaks)
    has_multiple_sentences = response_text.count('.') > 1
    has_transitions = any(t in response_text.lower() for t in ['first', 'second', 'also', 'however'])
    structure_score = 7.0 if has_multiple_sentences else 4.0
    if has_transitions:
        structure_score += 2
    structure_score = min(10, structure_score)

    # Detail/Reasoning quality
    has_explanation = any(e in response_text.lower() for e in ['because', 'since', 'reason', 'due to'])
    reasoning_score = 7.5 if has_explanation else 5.0

    # Creativity (uniqueness markers)
    creativity_markers = [
        'unique', 'interesting', 'however', 'paradox', 'interestingly', 'notably'
    ]
    creativity_score = min(10, (sum(1 for m in creativity_markers if m in response_text.lower()) + 1) * 2)

    # Overall score (equal-weighted average across all six criteria)
    overall_score = (
        accuracy_score * (1 / 6) +
        completeness_score * (1 / 6) +
        clarity_score * (1 / 6) +
        structure_score * (1 / 6) +
        reasoning_score * (1 / 6) +
        creativity_score * (1 / 6)
    )

    return {
        "accuracy": round(accuracy_score, 1),
        "completeness": round(completeness_score, 1),
        "clarity": round(clarity_score, 1),
        "structure": round(structure_score, 1),
        "reasoning_quality": round(reasoning_score, 1),
        "creativity": round(creativity_score, 1),
        "overall_score": round(overall_score, 1),
        "word_count": word_count,
    }

### 5.Evaluate Output Quality

In [76]:
def execute_all_strategies(user_query, temp=0.7, max_tokens=150, p=0.9, k=50):
    """Execute query across all available techniques with evaluation.

    Returns:
        list: Results with responses and quality scores
    """
    collected_results = []

    for strategy_name, strategy_func in STRATEGIES.items():
        prompt_text = strategy_func(user_query)
        response, time_taken = text_generation(
            prompt_text,
            temp=temp,
            max_tokens=max_tokens,
            p=p,
            k=k,
        )

        # Evaluate the response
        evaluation = evaluate_response(response, user_query)

        output_quantity = {
            "word_count": len(response.split()),
            "character_count": len(response),
            "line_count": response.count('\n') + 1,
            "token_estimate": len(response) // 4,
        }

        collected_results.append({
            "strategy": strategy_name,
            "prompt": prompt_text,
            "response": response,
            "inference_time_sec": time_taken,
            "evaluation": evaluation,
        })

    return collected_results

### 6. Bonus — Export Results to Markdown

In [73]:
def save_results_to_file(user_query, collected_results, output_file="prompt_comparison_report.md"):
    """Export results to Markdown with enhanced comparison metrics."""
    file_lines = [
        "# Prompt Engineering Studio - Comparison Report\n",
        f"**Query:** {user_query}\n",
        f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n",
        f"**Total Strategies Tested:** {len(collected_results)}\n",
    ]

    # Summary comparison table
    file_lines.append("\n## 📊 Quick Comparison\n\n")
    file_lines.append(
        "| Strategy | Word Count | Accuracy | Completeness | Clarity | Structure | "
        "Reasoning | Creativity | Overall | Time (s) |\n"
    )
    file_lines.append(
        "|----------|-----------|----------|--------------|---------|-----------|"
        "-----------|-----------|---------|---------|\n"
    )

    for entry in collected_results:
        eval_data = entry["evaluation"]
        file_lines.append(
            f"| {entry['strategy']} | {eval_data['word_count']} | {eval_data['accuracy']} | "
            f"{eval_data['completeness']} | {eval_data['clarity']} | "
            f"{eval_data['structure']} | {eval_data['reasoning_quality']} | "
            f"{eval_data['creativity']} | {eval_data['overall_score']} | "
            f"{entry['inference_time_sec']} |\n"
        )

    file_lines.append("\n---\n")

    # Detailed results
    for entry in collected_results:
        technique_name = entry["strategy"]
        prompt_text = entry["prompt"]
        output_text = entry["response"]
        exec_time = entry["inference_time_sec"]
        eval_data = entry["evaluation"]

        file_lines.append(f"\n## {technique_name}\n")
        file_lines.append(f"**Inference Time:** {exec_time}s\n")

        file_lines.append("### 📝 Prompt Used:\n")
        file_lines.append(f"```\n{prompt_text}\n```\n")

        file_lines.append("### 💬 Generated Output:\n")
        file_lines.append(f"{output_text}\n")

        file_lines.append("### 📈 Quality Evaluation:\n")
        file_lines.append(f"- **Accuracy:** {eval_data['accuracy']}/10\n")
        file_lines.append(f"- **Completeness:** {eval_data['completeness']}/10\n")
        file_lines.append(f"- **Clarity:** {eval_data['clarity']}/10\n")
        file_lines.append(f"- **Structure:** {eval_data['structure']}/10\n")
        file_lines.append(f"- **Reasoning Quality:** {eval_data['reasoning_quality']}/10\n")
        file_lines.append(f"- **Creativity:** {eval_data['creativity']}/10\n")
        file_lines.append(f"- **Overall Score:** {eval_data['overall_score']}/10\n")
        file_lines.append(f"- **Word Count:** {eval_data['word_count']}\n")

    # Write to file
    with open(output_file, "w", encoding="utf-8") as writer:
        writer.write("".join(file_lines))

    return output_file


## 7. Custom Query Testing

In [77]:
# ====== CUSTOMIZE THESE PARAMETERS ======
CUSTOM_QUERY = "What are renewable energy sources?"
CUSTOM_TEMPERATURE = 0.5  # Lower = more deterministic, Higher = more creative
CUSTOM_MAX_TOKENS = 200   # Longer responses
CUSTOM_TOP_P = 0.85
CUSTOM_TOP_K = 40
# ========================================

print(f"\n🔄 Running with custom parameters...")
print(f"Query: {CUSTOM_QUERY}")
print(f"Temperature: {CUSTOM_TEMPERATURE} (Lower = more focused, Higher = more creative)")
print(f"Max Tokens: {CUSTOM_MAX_TOKENS}\n")

custom_results = execute_all_strategies(
    CUSTOM_QUERY,
    temp=CUSTOM_TEMPERATURE,
    max_tokens=CUSTOM_MAX_TOKENS,
    p=CUSTOM_TOP_P,
    k=CUSTOM_TOP_K
)

print(f"✅ Generated {len(custom_results)} results with custom parameters")

# Export custom results
custom_export = save_results_to_file(CUSTOM_QUERY, custom_results, "custom_comparison_report.md")
print(f"✅ Custom results exported to: {custom_export}")


🔄 Running with custom parameters...
Query: What are renewable energy sources?
Temperature: 0.5 (Lower = more focused, Higher = more creative)
Max Tokens: 200

✅ Generated 6 results with custom parameters
✅ Custom results exported to: custom_comparison_report.md
